# Accès aux soins de santé - Rapport

## Population vivant à moins de 5 km d'un établissement de soins de santé

Ce rapport génère des visualisations liées à la population vivant dans un rayon de 5 km (qui correspond à environ 1 heure de marche) autour d'un établissement de santé, ou Formation Sanitaire (FOSA). Cette population est considérée comme ayant accès aux soins de santé. L' analyse est menée au niveau administratif du district ou de la commune, le cas échéant.

Le rapport se base sur les valeurs des paramètres utilisées dans le notebook de calcul de la population vivant non loin d'une FOSA, faisant partie du même pipeline. Il s'agit des choix suivants

* Le fichier des emplacements des FOSA. Ce sont des données spatiales (points) et peuvent être
    - chargées par l'utilisateur lors du lancement du pipeline
    - des données par défaut, provenant de DHIS2
* L'année de référence pour les effectifs de population à un découpage administratif très fin (100m)

Le rapport produit notamment les graphiques suivants, par unité administrative de niveau 2 (ADM2) :

* Nombre de FOSA
* Emplacement des FOSA, ainsi que les aires de couverture autour des FOSA
* Population totale
* Pourcentage de population avec accès aux soins de santé, présenté de manière continue (tel quel) et de manière catégorielle

## 1. Configuration

In [ ]:
rm(list = ls())

In [ ]:
# Global settings
options(scipen=999)
Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.setenv(GDAL_DATA = "/opt/conda/share/gdal")

In [ ]:
# Project paths
ROOT_PATH <- '~/workspace'
PROJECT_PATH <- file.path(ROOT_PATH, "pipelines/snt_healthcare_access")
CODE_PATH <- file.path(ROOT_PATH, 'code')
UTILS_PATH <- file.path(PROJECT_PATH, 'utils')

In [ ]:
# Load utils and bootstrap context
source(file.path(CODE_PATH, "snt_utils.r"))
source(file.path(CODE_PATH, "snt_report.r"))
source(file.path(CODE_PATH, "snt_palettes.r"))
source(file.path(UTILS_PATH, "snt_healthcare_access.r"))
source(file.path(UTILS_PATH, "snt_healthcare_access_report.r"))
setup_ctx <- bootstrap_healthcare_access_context(root_path = ROOT_PATH)

CONFIG_PATH <- setup_ctx$CONFIG_PATH
DATA_PATH <- setup_ctx$DATA_PATH
OUTPUT_DATA_PATH <- setup_ctx$OUTPUT_DATA_PATH
OUTPUT_PLOTS_PATH <- setup_ctx$OUTPUT_PLOTS_PATH
INTERMEDIATE_RESULTS_PATH <- setup_ctx$INTERMEDIATE_RESULTS_PATH

reticulate::py_config()$python

# Load SNT config
config_json <- tryCatch({ fromJSON(file.path(CONFIG_PATH, "SNT_config.json")) },
                        error = function(e) {
                          msg <- paste0("Error while loading configuration", conditionMessage(e))
                          cat(msg)
                          stop(msg)
                        })

pipeline_msg(glue("SNT configuration loaded from: {file.path(CONFIG_PATH, 'SNT_config.json')}"))

In [ ]:
#%% Config variables
# Set variables
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
ORG_UNITS_LEVEL <- config_json$SNT_CONFIG$ANALYTICS_ORG_UNITS_LEVEL
print(paste("Code pays: ", COUNTRY_CODE))

dhis2_dataset <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
healthcare_access_dataset <- config_json$SNT_DATASET_IDENTIFIERS$SNT_HEALTHCARE_ACCESS

In [ ]:
#%% Global variables
population_data_source <- "WorldPop"
admin_col <- "ADM2_ID"
admin_level <- "ADM2"

country_epsg_degrees <- 4326 # for plotting
country_epsg_meters <- 32630 # for creating the buffer areas

# column names
latitude_col <- "LATITUDE"
longitude_col <- "LONGITUDE"
coordinate_cols <- c(longitude_col, latitude_col) # longitude (x) first, latitude (y) second

### Paramètres

Importation des paramètres enregistrés lors de la dernière exécution des calculs.

In [ ]:
# Parameters
INPUT_FOSA_FILE <- NULL # Optional file (full path) with health facility locations
WORLDPOP_YEAR <- as.integer(format(Sys.Date(), "%Y")) - 1 # Year for WorldPop raster data

In [ ]:
parameters_file <- paste0(COUNTRY_CODE, "_parameters.json")

# Try to load parameters from dataset, if it exists
parameters <- tryCatch({
    get_latest_dataset_file_in_memory(healthcare_access_dataset, parameters_file)
}, error = function(e) {
    pipeline_msg(paste0("[WARNING] Parameters could not be loaded; using default values: ", conditionMessage(e)))
    NULL
})

In [ ]:
# Create label for FOSA data source (for plots)
if(is.null(parameters) || is.null(parameters[["INPUT_FOSA_FILE"]]) || is.na(parameters[["INPUT_FOSA_FILE"]]) || is.nan(parameters[["INPUT_FOSA_FILE"]])){
  fosa_data_source <- "DHIS2"
}else{
  fosa_data_source <- "Données de l'utilisateur"
}


## 2. Chargement et pré-processing des données à cartographier

**Les données utilisées**

* données administratives : fond de carte au niveau administratif ADM2
* données sur l'emplacement des FOSA
* données sur la population :
    * la population totale à haute résolution (raster)
    * la population totale agrégée par ADM2
    * la part de population avec/sans accès aux soins de santé, calculée par le notebook de calcul de l'accès aux soins, par ADM2

### Données administratives

In [ ]:
# spatial data filename
filename_spatial_units_data <- paste(COUNTRY_CODE, 'shapes.geojson', sep = '_')

# load as vector data
spatial_units_data <- tryCatch({ get_latest_dataset_file_in_memory(dhis2_dataset, paste0(COUNTRY_CODE, "_shapes.geojson")) }, 
                  error = function(e) {
                      msg <- paste("Error while loading DHIS2 Shapes data for: " , COUNTRY_CODE, conditionMessage(e))
                      cat(msg)
                      stop(msg)
                      })

spatial_units_data <- reproject_epsg(spatial_units_data, country_epsg_degrees)

### Données sur l'emplacement des FOSA

In [ ]:
# data on FOSA locations
fosa_vect_filtered_filename <- glue("{COUNTRY_CODE}_FOSA_filtered.gpkg")
fosa_vect_filtered <- st_read(file.path(INTERMEDIATE_RESULTS_PATH, fosa_vect_filtered_filename))

fosa_vect_filtered <- reproject_epsg(fosa_vect_filtered, country_epsg_degrees)

In [ ]:
# data on buffers around FOSA
coverage_vect_filename <- glue("{COUNTRY_CODE}_health_coverage_buffers.gpkg")
coverage_vect <- read_sf(file.path(INTERMEDIATE_RESULTS_PATH, coverage_vect_filename))

coverage_vect <- reproject_epsg(coverage_vect, country_epsg_degrees)

### Données sur la population

In [ ]:
# Import population raster, according to the reference year; format is {COUNTRY_CODE}_pop_{WORLDPOP_YEAR}_CN_100m_*.tif
wpop_raw_path <- file.path(DATA_PATH, "worldpop", "rasters")

if (!dir.exists(wpop_raw_path)) {
  stop(glue(
    "The {wpop_raw_path} directory, for WorldPop raster data, is missing."
  ))
}

wpop_pattern <- sprintf("%s_pop_%s_CN_100m_[[:alnum:]_-]+\\.tif$", tolower(COUNTRY_CODE), as.character(WORLDPOP_YEAR))
matching_files <- list.files(path = wpop_raw_path, pattern = wpop_pattern)

pop_path <- file.path(wpop_raw_path, matching_files[1])

pop_data <- tryCatch(
rast(pop_path),
error = function(e) stop(glue("Error while loading population raster: {conditionMessage(e)}"))
)
pipeline_msg(glue("Population raster data loaded: {pop_path}"))

In [ ]:
# population aggregated by ADM2
dt_filename_stem <- glue("{COUNTRY_CODE}_population_covered_health")
dt <- read_parquet(file.path(OUTPUT_DATA_PATH, glue("{dt_filename_stem}.parquet")))
setDT(dt)

In [ ]:
# make categories of covered population
set.seed(101010)
covered_k_means_breaks <- make_k_means_breaks(dt[["PCT_HEALTH_ACCESS"]], 3)
dt <- cut_to_categories(dt, "PCT_HEALTH_ACCESS", "CAT_COVERED", covered_k_means_breaks, num_decimals = 1, suffix = "%")

In [ ]:
plot_data <- merge(spatial_units_data, dt, by = c("ADM1_ID", "ADM1_NAME", "ADM2_ID", "ADM2_NAME"), all.x = TRUE)

## 3. Génération des graphiques de résultat

### Nombre de FOSA par ADM2

Cette carte montre, **par ADM2, le nombre total d'établissements de santé**, peu importe leur type.

In [ ]:
fosa_per_admin_plot <- make_fosa_choropleth_map(
  spatial_data = spatial_units_data,
  fosa_data = fosa_vect_filtered,
  epsg_value_degrees = country_epsg_degrees,
  spatial_data_id_colname = admin_col,
  low_color = "#f1e5bd",
  high_color = "#8d6c00",
  plot_title = "Nombre de FOSA",
  plot_subtitle = NULL,
  plot_caption = glue("Données: {fosa_data_source}"))


In [ ]:
fosa_per_admin_plot

In [ ]:
# save as .png file
fosa_per_admin_filename <- glue::glue("{COUNTRY_CODE}_{fosa_data_source}_{admin_level}_num_FOSA.png")
ggsave(file.path(OUTPUT_PLOTS_PATH, fosa_per_admin_filename), plot = fosa_per_admin_plot)

### Emplacement et couverture territoriale des FOSA

La carte suivante montre **l'emplacement exact des FOSA** sur le territoire, ainsi que **les zones de couverture de 5 km autour de chaque FOSA**.

In [ ]:
coverage_plot <- make_overlaid_sf_plot(
  admin_unit_vect=spatial_units_data,
  points_sf_vect=fosa_vect_filtered,
  buffer_vect=coverage_vect,
  epsg_value_degrees=country_epsg_degrees,
  plot_title=glue("Couverture de l'accès aux soins de santé ({COUNTRY_CODE})"),
  plot_caption=glue("Données\nFOSA: {fosa_data_source}\nPopulation: {population_data_source}")
)

In [ ]:
coverage_plot

In [ ]:
# save as .png file
timestamp <- format(Sys.time(), "%Y-%m-%d_%H%M%S")
coverage_plot_filename <- glue("{COUNTRY_CODE}_coverage_plot_{timestamp}.png")
invisible(ggsave(filename=file.path(OUTPUT_PLOTS_PATH, coverage_plot_filename), plot=coverage_plot))
pipeline_msg(glue("{COUNTRY_CODE} Coverage map saved: {file.path(OUTPUT_PLOTS_PATH, coverage_plot_filename)}"))

### Population totale

#### Population haute résolution

Cette carte reprend **les données d'entrée sur la population**, utilisées dans les calculs d'accès aux soins. Ce sont des données de haute résolution, appelées `raster`, qui expriment des éstimations démographiques de la population, maillées à une résolution de 100 mètres.

In [ ]:
population_raster_plot <- plot_raster_with_boundaries(
    input_raster = pop_data,
    input_vector = spatial_units_data,
    epsg_value_degrees = country_epsg_degrees,
    low_color = "#e9d9f4",
    high_color = "#2e0044",
    plot_title = "Population (haute résolution)",
    plot_subtitle = {WORLDPOP_YEAR},
    plot_caption = glue("Données: {population_data_source}")
)

In [ ]:
population_raster_plot

In [ ]:
population_raster_filename <- glue::glue("{COUNTRY_CODE}_{population_data_source}_{admin_level}_pop_hires.png")

ggsave(file.path(OUTPUT_PLOTS_PATH, population_raster_filename), population_raster_plot)

#### Population aggrégée par ADM2

La carte ci-dessous présente **ces mêmes données de population, mais agrégées par unité administrative de niveau 2**, dans le cadre du pipeline

In [ ]:
total_population_plot <- make_snt_choropleth_map(
    input_data = plot_data,
    target_colname = "POP_TOTAL",
    low_color = "#f7fbff",
    high_color = "#08306b",
    plot_title = "Population totale",
    plot_subtitle = {WORLDPOP_YEAR},
    plot_caption = glue("Données: {population_data_source}"),
    legend_title = NULL
)

In [ ]:
total_population_plot

In [ ]:
# save as .png file
total_population_filename <- glue::glue("{COUNTRY_CODE}_{population_data_source}_{admin_level}_pop_total.png")
ggsave(file.path(OUTPUT_PLOTS_PATH, total_population_filename), plot = total_population_plot)

### Population couverte et non couverte

Le but principal de ce pipeline est d'établir, par ADM2, quelle part de la population se trouve dans un rayon de 5 km autour d'un établissement de santé, et peut donc être considérée comme ayant accès aux soins de santé.

Les deux cartes suivantes présentent donc le pourcentage de population "couverte"par l'accès aux soins de santé, de deux manières:
    * ce pourcentage tel quel
    * de façon catégorielle, où les catégories sont établies pour que toutes les catégories comprennent des effectifs égaux

Cette première carte présente, **par unité administrative, le pourcentage de population avec accès aux soins de santé**.

In [ ]:
pop_covered_plot <- make_snt_choropleth_map(
    input_data = plot_data,
    target_colname = "PCT_HEALTH_ACCESS",
    low_color = "#610228",
    high_color = "#f0dbe2",
    plot_title = "Population avec accès aux soins de santé (%)",
    plot_subtitle = {WORLDPOP_YEAR},
    plot_caption = glue("Données\nFOSA: {fosa_data_source}\nPopulation: {population_data_source}"),
    legend_title = NULL
)

In [ ]:
pop_covered_plot

In [ ]:
# Save as .png file
pop_covered_plot_filename <- glue("{COUNTRY_CODE}_{admin_level}_{WORLDPOP_YEAR}_pop_health_covered_plot.png")
ggsave(file.path(OUTPUT_PLOTS_PATH, pop_covered_plot_filename), plot = pop_covered_plot)

La seconde carte montre, **ces mêmes données que la carte précédente, mais de manière catégorielle**, soit les données sont présentées sous forme de tranches de pourcentages. Ces tranches ont été calculées utilisant la méthode k-means, afin de préserver les éventuels groupements (clusters) présents dans la série des données.

In [ ]:
cat_covered_plot <- make_snt_categorical_map(
    input_data = plot_data,
    target_colname = "CAT_COVERED",
    color_vector = cat_healthcare_covered_palette,
    plot_title = "Population avec accès aux soins de santé",
    plot_subtitle = {WORLDPOP_YEAR},
    plot_caption = glue("Données\nFOSA: {fosa_data_source}\nPopulation: {population_data_source}"),
)

In [ ]:
cat_covered_plot

In [ ]:
# save as .png file
cat_covered_plot_filename <- glue("{COUNTRY_CODE}_{admin_level}_{WORLDPOP_YEAR}_cat_pop_health_covered_plot.png")
ggsave(file.path(OUTPUT_PLOTS_PATH, cat_covered_plot_filename), plot = cat_covered_plot)